# Feature Store & Hashing


Every model in stages 3-9 needs features. The shape of the feature dict — which user attributes, which item attributes, which contextual cues — is the same whether the model is ALS (REC:03), a Two-Tower retriever (REC:04), or an LLM re-ranker (REC:06). This notebook builds the *feature store* that abstracts over all of them. The model code does not care whether the features come from a Pandas dataframe in a notebook or a Redis lookup in production. That abstraction is what makes REC:07 possible.

There are two design choices in this stage that will pay off for the rest of the course: (1) using **feature hashing** instead of learned vocabularies, so the serving vocabulary cannot drift from the training one; and (2) implementing a minimal TF-IDF in pure NumPy so the same `FeatureStore` works both at training time (Python) and at serving time (FastAPI) without depending on whatever pickle format sklearn wrote two years ago.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens
from notebooks.recsys.features import (
    FeatureStore, MovieFeatures, UserFeatures,
    hash_bucket, hash_vector, tfidf_fit, tfidf_transform,
)


## Feature hashing

A naive encoder for a categorical variable $x \in V$ builds a vocabulary $\{v_1, \ldots, v_K\}$ and assigns a learned embedding $\mathbf{e}_{v_k} \in \mathbb{R}^d$ to each $v_k$. The vocabulary grows with data. At serving time, a token you have never seen (`new_movie`) has no embedding, and the model silently degrades.

**Feature hashing** (Weinberger, Das, Langford, Smola, Attenberg 2009) sidesteps this entirely. Define $\phi: V \to \{0, 1, \ldots, 2^b - 1\}$ as a uniform random hash, and represent $x$ as a one-hot of size $2^b$ at index $\phi(x)$. With $K = |V|$ distinct tokens and $B = 2^b$ buckets, the expected collision count is

$$\mathbb{E}[\#\{\text{collisions}\}] = K - B \left(1 - \left(1 - \frac{1}{B}\right)^K \right) \approx K - B \cdot (1 - e^{-K / B}).$$

Pick $B / K \approx 4$, and the collision rate is below $\sim 1\%$. Below, we use $b = 16$, so $B = 65536$ — comfortably above $|V|$ for MovieLens's $\sim 1600$ movies.

The crucial trick is the *signed* variant: assign each token $x$ a pseudorandom sign $s(x) \in \{-1, +1\}$ and build $\phi_s(x) = s(x) \cdot \mathbf{1}_{\phi(x)}$. The pairwise inner products are then unbiased estimators of the un-hashed inner products:

$$\boxed{\, \mathbb{E}\big[\langle \phi_s(x), \phi_s(y) \rangle\big] = \langle x, y \rangle. \,}$$

This is the *count-sketch* trick (Charikar, Chen, Farach-Colton, 2002). It is what makes feature hashing usable as embedding-table input for the Two-Tower model in REC:04 and not just for linear models.


In [ ]:
dim = 256
# Same token twice should add up; different tokens should not interfere in expectation
x = hash_vector(["alice", "bob"], dim=dim)
y = hash_vector(["alice", "bob"], dim=dim)
z = hash_vector(["alice", "carol"], dim=dim)
print(f"<x, y> = {x @ y:.2f}   (expect 2.00)")
print(f"<x, z> = {x @ z:.2f}   (expect 1.00)")


The signed ($\pm 1$) variant above passes the unbiasedness test on expectation; the unsigned variant puts $\langle x, y \rangle$ on top of a $\Theta(\sqrt{K})$ collision background.


## The minimal TF-IDF

We need *some* text features for the content-based baseline in REC:03 and for the LLM re-ranker's RAG context in REC:06. Rather than pull in sklearn's `TfidfVectorizer` (a perfectly good library with a pickle that will one day be incompatible with our package), we implement a 30-line TF-IDF that exposes every coefficient:

$$\mathrm{tf}(t, d) = \frac{c(t, d)}{\|c(\cdot, d)\|_1}, \qquad \mathrm{idf}(t) = \log\frac{1 + N}{1 + \mathrm{df}(t)} + 1, \qquad \mathrm{tfidf} = \mathrm{tf} \cdot \mathrm{idf}.$$

The $+1$ smoothing in the IDF is what sklearn does. We L2-normalize the rows so the cosine similarity $\langle \mathrm{tfidf}_i, \mathrm{tfidf}_j \rangle / \|\cdot\|^2$ is a well-behaved content-similarity score.


In [ ]:
corpus = ["the quick brown fox jumps",
           "the lazy dog sleeps quietly",
           "the quick fox and the lazy fox race"]
tfidf = tfidf_fit(corpus)
print(f"vocab size: {len(tfidf['vocab'])}")
mat = tfidf_transform(tfidf, corpus)
print(f"similarity matrix:")
print(np.round(mat @ mat.T, 2))  # diagonal should be 1 (normalized) -- approx


## The `FeatureStore`

`FeatureStore` wraps a `Dataset` and exposes:

* `get_user_features(user_id) -> UserFeatures`
* `get_item_features(item_id) -> MovieFeatures`
* `content_matrix() -> (TF-IDF matrix, item_ids)`
* `hash_user_history(user_id, dim)` — a signed hashed bag of recent item ids for downstream deep models

The store is the place that the REC:07 FastAPI service hits when evaluating a model. Keep its public surface minimal; never let a model import `Dataset` directly. That separation is the difference between a project that runs once and one that deploys.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
store = FeatureStore(ds)

# Pick one popular movie to inspect
popular_iid = ds.ratings.groupby('item_id').size().idxmax()
mf = store.get_item_features(popular_iid)
print(f"item:           {mf.title}")
print(f"genres:        {mf.genres}")
print(f"popularity:    {mf.popularity:.2f}  (log1p of {mf.n_ratings} ratings)")
print(f"mean_rating:   {mf.mean_rating:.2f}")
print(f"TF-IDF dim:    {mf.tfidf.shape}")


In [ ]:
# Cold-start user lookups return a user_features object with mean rating = NaN
uf = store.get_user_features(ds.ratings['user_id'].iloc[0])
print(f"user_id:        {uf.user_id}")
print(f"activity:       {uf.activity}")
print(f"log_activity: {uf.log_activity:.2f}")
print(f"mean_rating:   {uf.mean_rating:.2f}")
print(f"top genres:    {uf.top_genres}")
print(f"recent {len(uf.recent_item_ids)} item ids")


## Online / offline parity

The `FeatureStore` exists to make *online/offline parity* automatic. The principle:

$$\forall (u, i): \quad \phi_{train}(u, i) \stackrel{!}{=} \phi_{serve}(u, i).$$

Any drift between the feature vector used to train the ranker and the feature vector used to score online requests is a model bug — sometimes invisible, sometimes catastrophic. The classic offenders are: (1) **vocabulary drift** when the feature encoder is retrained; (2) **time leakage** when "user activity in last 30 days" is computed differently for offline vs online; and (3) **NaN handling**: training treats unknown features as zero; serving treats them as zero too, but only if you absolutely know it.

Our `FeatureStore` removes the first offender by being stateless — every call recomputes features from the underlying `Dataset` / `ratings` table — and never persists vocabularies. The dispatcher in REC:07 will use the exact same `FeatureStore` object for training-time evaluation and online serving. The drift offenders (2) and (3) become a REC:08 problem rather than a structural one.


## Content similarity as a baseline signal

Below is the cosine similarity of the TF-IDF vectors of the top-200 movies by rating count, clustered by genres. This is the matrix the content-based ranker in REC:03 will sample from.


In [ ]:
# Top-200 by rating count
top_iids = ds.ratings.groupby('item_id').size().nlargest(200).index.tolist()
M = np.stack([store.vectorize_item(i) for i in top_iids])
M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
S = M @ M.T

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(S, cmap='viridis')
ax.set_title('Cosine similarity of TF-IDF, top-200 MovieLens items')
ax.set_xlabel('item #'); ax.set_ylabel('item #')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()


**Observation.** There is sensible block structure here — comedies look more like comedies than like horror. The cosine values are hydrated above zero because the corpus has shared stop words even after our tiny normalization. A real production content pipeline would use sentence embeddings (REC:06) or pretrained transformers. The point of this matrix is to give the *first* model a fighting chance.


## Caveats and link forward

What `FeatureStore` deliberately does not do:

1. **No streaming updates.** It reads the `Dataset.ratings` table once and never refreshes. MovieLens 25M does not change under us, but in production user activity changes by the second. The streaming solution is Feast (cloud) or Redis + a Lambda updating per-item aggregates. The `FeatureStore` interface is structured deliberately so that those backends are drop-in replacements: subclass and override `get_user_features` / `get_item_features`.
2. **No cross-features.** Real production systems lean heavily on combinations like `genre × user_decade` that are seen thousands of times. We will build cross-features in REC:05 (Wide&Deep) instead, with the hash trick keeping the coefficient count bounded.
3. **No privacy.** The hashed bag leaks counts trivially: hash `item_42` and read the bucket. That is a problem for federated learning; it is fine for our single-machine demo.

Next: REC:03 trains the first real models — ALS for collaborative filtering and a content-based fallback — and builds the metrics the rest of the course will use to compare models.
